# 🔐 Zero Trust Security Architecture (ZTSA) for Securing Cloud Environments
## MSc Cyber Security — Ulster University Belfast

---

### Project Overview
This tool implements a **Zero Trust Security Architecture (ZTSA)** framework for assessing and securing cloud environments.

Based on **NIST SP 800-207** and **NCSC Cloud Security Principles**, it evaluates cloud users and devices across **15 security factors** and makes real-time access control decisions.

**Core Principle:** *'Never Trust, Always Verify — Especially in the Cloud'*

| Section | Description |
|---------|-------------|
| **Cell 1** | Install & Import Libraries |
| **Cell 2** | ZTSA Scoring Engine (15 Security Factors) |
| **Cell 3** | Cloud User Dataset (10 Users — AWS, Azure, GCP) |
| **Cell 4** | Data Processing & Analysis |
| **Cell 5** | Visualisation Charts 1, 2 & 3 |
| **Cell 6** | Visualisation Charts 4, 5 & 6 |
| **Cell 7** | Live Demo — Test Any Cloud User |
| **Cell 8** | Final Summary Report |

---
**References:**
- Rose et al., NIST SP 800-207, 2020
- NCSC Cloud Security Principles, 2023
- IBM Cost of a Data Breach Report, 2024
- Gartner Cloud Security Report, 2024

---
## 📦 Cell 1 — Install & Import Libraries

In [ ]:
# ── CELL 1: Install and Import All Required Libraries ─────────────────────────
# Run this cell first to ensure all libraries are installed

import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'pandas', 'numpy', 'matplotlib',
                       'seaborn', '--quiet'])

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Colour Scheme (Ocean/Cloud Theme) ─────────────────────────────────────────
DEEP     = '#065A82'   # Deep blue
TEAL     = '#1C7293'   # Teal
MID      = '#028090'   # Mid teal
MIDNIGHT = '#21295C'   # Midnight navy
GREEN    = '#00C48C'   # Access granted
RED      = '#E63946'   # Access denied
AMBER    = '#FFB347'   # Verify identity
GREY     = '#8892A4'   # Neutral
WHITE    = '#FFFFFF'
OFFWHITE = '#F4F8FB'

# ── Visual Style ──────────────────────────────────────────────────────────────
plt.rcParams['figure.facecolor'] = OFFWHITE
plt.rcParams['axes.facecolor']   = WHITE
plt.rcParams['font.family']      = 'DejaVu Sans'
sns.set_theme(style='whitegrid', palette='muted')

print('=' * 60)
print('  🔐  ZTSA CLOUD SECURITY ANALYSER')
print('  MSc Cyber Security — Ulster University Belfast')
print('  Based on NIST SP 800-207 & NCSC Cloud Principles')
print('=' * 60)
print('✅ All libraries imported successfully!')
print(f'  pandas  {pd.__version__}')
print(f'  numpy   {np.__version__}')
print(f'  seaborn {sns.__version__}')

---
## ⚙️ Cell 2 — ZTSA Scoring Engine

The scoring engine evaluates **15 cloud security factors** across 5 categories:

| Category | Factors | NIST SP 800-207 Tenet |
|----------|---------|----------------------|
| Identity | MFA, SSO, Certificate | Tenet 1 — Verify Explicitly |
| Device | Patched, Encrypted, Antivirus, MDM | Tenet 2 — Device Compliance |
| Cloud Config | S3 Public, IAM Privilege, CloudTrail | Tenet 3 — Least Privilege |
| Network | VPN, Known IP, Geo-location | Tenet 4 — Assume Breach |
| Behaviour | Failed Logins, Unusual Hours, Data Volume | Tenet 5 — Continuous Verification |

In [ ]:
# ── CELL 2: ZTSA Scoring Engine ───────────────────────────────────────────────
# Based on NIST SP 800-207 Zero Trust Architecture tenets
# and NCSC 14 Cloud Security Principles

def ztsa_score(user):
    """
    Calculates a Zero Trust trust score for a cloud user.
    Starts at 100 and deducts points for each security risk found.
    Returns score, access decision, risk level and detailed findings.
    
    Parameters: user (dict) — cloud user security attributes
    Returns: dict — score, decision, risk, findings, categories
    """
    score    = 100
    findings = []
    risks    = []

    # ══════════════════════════════════════════════════════════
    # CATEGORY 1: IDENTITY VERIFICATION
    # NIST SP 800-207 Tenet 1 — All access requests authenticated
    # ══════════════════════════════════════════════════════════
    if not user['mfa_enabled']:
        score -= 30  # Highest penalty — MFA blocks 99.9% of account attacks
        findings.append('❌ MFA not enabled — critical identity risk')
        risks.append('Identity')
    else:
        findings.append('✅ MFA enabled')

    if not user['sso_configured']:
        score -= 10
        findings.append('❌ SSO not configured — centralised auth missing')
        risks.append('Identity')
    else:
        findings.append('✅ SSO configured')

    if not user['valid_certificate']:
        score -= 15
        findings.append('❌ No valid device certificate')
        risks.append('Identity')
    else:
        findings.append('✅ Valid device certificate')

    # ══════════════════════════════════════════════════════════
    # CATEGORY 2: DEVICE HEALTH
    # NIST SP 800-207 Tenet 2 — Device integrity must be verified
    # ══════════════════════════════════════════════════════════
    if not user['device_patched']:
        score -= 20
        findings.append('❌ Device not patched — known vulnerabilities exposed')
        risks.append('Device')
    else:
        findings.append('✅ Device fully patched')

    if not user['device_encrypted']:
        score -= 10
        findings.append('❌ Device not encrypted')
        risks.append('Device')
    else:
        findings.append('✅ Device encrypted')

    if not user['antivirus_active']:
        score -= 10
        findings.append('❌ Antivirus not active')
        risks.append('Device')
    else:
        findings.append('✅ Antivirus active')

    if not user['mdm_enrolled']:
        score -= 10
        findings.append('❌ Device not enrolled in MDM')
        risks.append('Device')
    else:
        findings.append('✅ MDM enrolled')

    # ══════════════════════════════════════════════════════════
    # CATEGORY 3: CLOUD CONFIGURATION
    # NCSC Cloud Security Principle 9 — Secure user management
    # Gartner: 99% of cloud failures are misconfigurations
    # ══════════════════════════════════════════════════════════
    if user['s3_public_bucket']:
        score -= 25  # Critical — public cloud storage is a data breach risk
        findings.append('❌ PUBLIC cloud storage bucket detected — CRITICAL')
        risks.append('Cloud Config')
    else:
        findings.append('✅ Cloud storage buckets private')

    if user['iam_overprivileged']:
        score -= 20
        findings.append('❌ IAM over-privileged — violates least privilege principle')
        risks.append('Cloud Config')
    else:
        findings.append('✅ IAM least privilege enforced')

    if not user['cloudtrail_enabled']:
        score -= 15
        findings.append('❌ Cloud audit logging disabled — no audit trail')
        risks.append('Cloud Config')
    else:
        findings.append('✅ Cloud audit logging enabled')

    # ══════════════════════════════════════════════════════════
    # CATEGORY 4: NETWORK SECURITY
    # NIST SP 800-207 Tenet 4 — Assume breach, verify network
    # ══════════════════════════════════════════════════════════
    if not user['vpn_connected']:
        score -= 15
        findings.append('❌ Not connected via VPN')
        risks.append('Network')
    else:
        findings.append('✅ VPN connected')

    if not user['known_ip']:
        score -= 10
        findings.append('⚠️  Connection from unknown IP address')
        risks.append('Network')
    else:
        findings.append('✅ Known IP address')

    if user['geo_anomaly']:
        score -= 20
        findings.append('⚠️  Geo-location anomaly — unusual country detected')
        risks.append('Network')

    # ══════════════════════════════════════════════════════════
    # CATEGORY 5: BEHAVIOURAL ANALYSIS
    # NIST SP 800-207 Tenet 5 — Continuous monitoring
    # ══════════════════════════════════════════════════════════
    if user['failed_logins'] > 3:
        score -= 20
        findings.append(f'⚠️  {user["failed_logins"]} failed login attempts detected')
        risks.append('Behaviour')
    elif user['failed_logins'] > 0:
        score -= 5
        findings.append(f'⚠️  {user["failed_logins"]} failed login attempt(s)')

    if user['unusual_hours']:
        score -= 10
        findings.append('⚠️  Access at unusual hours detected')
        risks.append('Behaviour')

    if user['high_data_volume']:
        score -= 15
        findings.append('⚠️  Unusually high data volume — possible exfiltration')
        risks.append('Behaviour')

    # ── Cap score between 0 and 100 ────────────────────────────────────────────
    score = max(0, min(100, score))

    # ── Access Decision ────────────────────────────────────────────────────────
    if score >= 70:
        decision = '✅ ACCESS GRANTED'
        color    = GREEN
        risk     = 'LOW RISK'
    elif score >= 40:
        decision = '⚠️  VERIFY IDENTITY'
        color    = AMBER
        risk     = 'MEDIUM RISK'
    else:
        decision = '🚨 ACCESS DENIED'
        color    = RED
        risk     = 'HIGH RISK'

    return {
        'score':    score,
        'decision': decision,
        'color':    color,
        'risk':     risk,
        'findings': findings,
        'risks':    list(set(risks))
    }

print('✅ ZTSA Scoring Engine loaded!')
print('   15 security factors across 5 categories')
print('   Based on: NIST SP 800-207 & NCSC Cloud Principles')
print('   Decision thresholds: ≥70% Granted | 40-69% Verify | <40% Denied')

---
## 🗄️ Cell 3 — Cloud User Dataset

10 simulated cloud users representing realistic roles across **AWS, Azure and GCP** environments.

In [ ]:
# ── CELL 3: Cloud User Dataset ────────────────────────────────────────────────
# 10 simulated cloud users based on realistic enterprise cloud roles
# Each user has a different risk profile across the 15 security factors

cloud_users = [
    {
        'id': 1, 'name': 'Cloud Admin',
        'role': 'AWS Root Admin', 'platform': 'AWS',
        # Identity
        'mfa_enabled': False, 'sso_configured': False, 'valid_certificate': False,
        # Device
        'device_patched': False, 'device_encrypted': False,
        'antivirus_active': True, 'mdm_enrolled': False,
        # Cloud Config
        's3_public_bucket': True, 'iam_overprivileged': True, 'cloudtrail_enabled': False,
        # Network
        'vpn_connected': False, 'known_ip': False, 'geo_anomaly': True,
        # Behaviour
        'failed_logins': 0, 'unusual_hours': True, 'high_data_volume': True
    },
    {
        'id': 2, 'name': 'Developer',
        'role': 'Azure DevOps Engineer', 'platform': 'Azure',
        'mfa_enabled': True, 'sso_configured': True, 'valid_certificate': True,
        'device_patched': True, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': True,
        's3_public_bucket': False, 'iam_overprivileged': False, 'cloudtrail_enabled': True,
        'vpn_connected': True, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 0, 'unusual_hours': False, 'high_data_volume': False
    },
    {
        'id': 3, 'name': 'Remote Worker',
        'role': 'Office 365 Finance', 'platform': 'Azure',
        'mfa_enabled': True, 'sso_configured': True, 'valid_certificate': False,
        'device_patched': False, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': False,
        's3_public_bucket': False, 'iam_overprivileged': False, 'cloudtrail_enabled': True,
        'vpn_connected': False, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 1, 'unusual_hours': True, 'high_data_volume': False
    },
    {
        'id': 4, 'name': 'Contractor',
        'role': 'GCP External Vendor', 'platform': 'GCP',
        'mfa_enabled': False, 'sso_configured': False, 'valid_certificate': False,
        'device_patched': False, 'device_encrypted': False,
        'antivirus_active': False, 'mdm_enrolled': False,
        's3_public_bucket': True, 'iam_overprivileged': True, 'cloudtrail_enabled': False,
        'vpn_connected': False, 'known_ip': False, 'geo_anomaly': True,
        'failed_logins': 8, 'unusual_hours': True, 'high_data_volume': True
    },
    {
        'id': 5, 'name': 'Data Scientist',
        'role': 'AWS S3 Analytics', 'platform': 'AWS',
        'mfa_enabled': True, 'sso_configured': True, 'valid_certificate': True,
        'device_patched': True, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': True,
        's3_public_bucket': False, 'iam_overprivileged': True, 'cloudtrail_enabled': True,
        'vpn_connected': True, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 0, 'unusual_hours': False, 'high_data_volume': True
    },
    {
        'id': 6, 'name': 'Finance Staff',
        'role': 'Azure Finance Portal', 'platform': 'Azure',
        'mfa_enabled': True, 'sso_configured': True, 'valid_certificate': True,
        'device_patched': True, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': True,
        's3_public_bucket': False, 'iam_overprivileged': False, 'cloudtrail_enabled': True,
        'vpn_connected': True, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 0, 'unusual_hours': False, 'high_data_volume': False
    },
    {
        'id': 7, 'name': 'IT Support',
        'role': 'Multi-Cloud Admin', 'platform': 'Multi-Cloud',
        'mfa_enabled': True, 'sso_configured': False, 'valid_certificate': True,
        'device_patched': True, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': True,
        's3_public_bucket': False, 'iam_overprivileged': True, 'cloudtrail_enabled': True,
        'vpn_connected': True, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 2, 'unusual_hours': True, 'high_data_volume': False
    },
    {
        'id': 8, 'name': 'CEO',
        'role': 'Office 365 Executive', 'platform': 'Azure',
        'mfa_enabled': True, 'sso_configured': True, 'valid_certificate': True,
        'device_patched': True, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': True,
        's3_public_bucket': False, 'iam_overprivileged': False, 'cloudtrail_enabled': True,
        'vpn_connected': True, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 0, 'unusual_hours': False, 'high_data_volume': False
    },
    {
        'id': 9, 'name': 'Intern',
        'role': 'AWS Limited Access', 'platform': 'AWS',
        'mfa_enabled': False, 'sso_configured': False, 'valid_certificate': False,
        'device_patched': False, 'device_encrypted': False,
        'antivirus_active': False, 'mdm_enrolled': False,
        's3_public_bucket': False, 'iam_overprivileged': False, 'cloudtrail_enabled': False,
        'vpn_connected': False, 'known_ip': False, 'geo_anomaly': False,
        'failed_logins': 4, 'unusual_hours': False, 'high_data_volume': False
    },
    {
        'id': 10, 'name': 'Security Analyst',
        'role': 'SIEM Cloud Monitor', 'platform': 'Multi-Cloud',
        'mfa_enabled': True, 'sso_configured': True, 'valid_certificate': True,
        'device_patched': True, 'device_encrypted': True,
        'antivirus_active': True, 'mdm_enrolled': True,
        's3_public_bucket': False, 'iam_overprivileged': False, 'cloudtrail_enabled': True,
        'vpn_connected': True, 'known_ip': True, 'geo_anomaly': False,
        'failed_logins': 0, 'unusual_hours': False, 'high_data_volume': False
    },
]

print(f'✅ Cloud User Dataset loaded: {len(cloud_users)} users')
print(f'   Platforms: AWS, Azure, GCP, Multi-Cloud')
print(f'   Roles: Admin, Developer, Remote Worker, Contractor, Data Scientist...')

---
## 📊 Cell 4 — Run ZTSA Analysis & Process Results

In [ ]:
# ── CELL 4: Run ZTSA Analysis on All Cloud Users ──────────────────────────────

print('Running ZTSA analysis...')
print()

results = []
for user in cloud_users:
    result = ztsa_score(user)
    results.append({
        'id':       user['id'],
        'name':     user['name'],
        'role':     user['role'],
        'platform': user['platform'],
        'score':    result['score'],
        'decision': result['decision'],
        'risk':     result['risk'],
        'color':    result['color'],
        'risks':    result['risks'],
        'findings': result['findings']
    })

df = pd.DataFrame(results)

# ── Compliance Analysis ────────────────────────────────────────────────────────
factors = [
    ('mfa_enabled',        'MFA Enabled'),
    ('sso_configured',     'SSO Configured'),
    ('valid_certificate',  'Valid Certificate'),
    ('device_patched',     'Device Patched'),
    ('device_encrypted',   'Device Encrypted'),
    ('antivirus_active',   'Antivirus Active'),
    ('mdm_enrolled',       'MDM Enrolled'),
    ('cloudtrail_enabled', 'CloudTrail Logging'),
    ('vpn_connected',      'VPN Connected'),
    ('known_ip',           'Known IP Address'),
]

compliance = {}
for key, label in factors:
    rate = sum(u[key] for u in cloud_users) / len(cloud_users) * 100
    compliance[label] = round(rate, 1)

# ── Risk category flags (reversed for public bucket, iam, geo anomaly, hours)
risk_flags = {
    's3_public_bucket':    sum(1 for u in cloud_users if u['s3_public_bucket']),
    'iam_overprivileged':  sum(1 for u in cloud_users if u['iam_overprivileged']),
    'geo_anomaly':         sum(1 for u in cloud_users if u['geo_anomaly']),
    'unusual_hours':       sum(1 for u in cloud_users if u['unusual_hours']),
    'high_data_volume':    sum(1 for u in cloud_users if u['high_data_volume']),
}

# ── Print Results Table ────────────────────────────────────────────────────────
print('=' * 72)
print('   🔐  ZTSA CLOUD SECURITY — ACCESS CONTROL RESULTS')
print('=' * 72)
print(f"{'ID':<4} {'Name':<18} {'Platform':<14} {'Score':>6} {'Decision':<25} {'Risk':<12}")
print('-' * 72)
for r in results:
    clean_decision = r['decision'].replace('✅ ','').replace('⚠️  ','').replace('🚨 ','')
    print(f"{r['id']:<4} {r['name']:<18} {r['platform']:<14} {r['score']:>5}%  {clean_decision:<25} {r['risk']:<12}")
print('=' * 72)

granted = len(df[df['score'] >= 70])
verify  = len(df[(df['score'] >= 40) & (df['score'] < 70)])
denied  = len(df[df['score'] < 40])

print(f'\n  ✅ Access Granted : {granted} users')
print(f'  ⚠️  Verify Identity : {verify} users')
print(f'  🚨 Access Denied  : {denied} users')
print(f'\n  Average Trust Score : {df["score"].mean():.1f}%')
print(f'  Highest Score       : {df["score"].max()}% ({df.loc[df["score"].idxmax(),"name"]})')
print(f'  Lowest Score        : {df["score"].min()}% ({df.loc[df["score"].idxmin(),"name"]})')
print(f'  Score Range         : {df["score"].max() - df["score"].min()} points')
print('=' * 72)

---
## 📈 Cell 5 — Visualisation: Charts 1, 2 & 3

In [ ]:
# ── CELL 5: Charts 1, 2 & 3 ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('ZTSA Cloud Security — Access Control Dashboard',
             fontsize=14, fontweight='bold', color=MIDNIGHT, y=1.02)

# ── CHART 1: Trust Score per Cloud User ───────────────────────────────────────
bar_colors = []
for score in df['score']:
    if score >= 70:   bar_colors.append(GREEN)
    elif score >= 40: bar_colors.append(AMBER)
    else:             bar_colors.append(RED)

bars = axes[0].bar(df['name'], df['score'],
                   color=bar_colors, edgecolor='white',
                   linewidth=1.5, width=0.65)
axes[0].axhline(y=70, color=GREEN, linestyle='--',
                linewidth=1.5, label='Granted (≥70%)', alpha=0.8)
axes[0].axhline(y=40, color=AMBER, linestyle='--',
                linewidth=1.5, label='Verify (≥40%)', alpha=0.8)
axes[0].set_title('Chart 1: ZTSA Trust Score per User',
                  fontweight='bold', color=MIDNIGHT, pad=12)
axes[0].set_ylabel('Trust Score (%)')
axes[0].set_ylim(0, 118)
axes[0].tick_params(axis='x', rotation=45)
axes[0].legend(fontsize=8)
for bar, val in zip(bars, df['score']):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1.5,
                 f'{val}%', ha='center',
                 fontsize=8, fontweight='bold', color=MIDNIGHT)

# ── CHART 2: Cloud Platform Risk ──────────────────────────────────────────────
platform_scores = df.groupby('platform')['score'].mean().sort_values()
p_colors = [GREEN if s >= 70 else AMBER if s >= 40 else RED
            for s in platform_scores.values]
bars2 = axes[1].barh(platform_scores.index, platform_scores.values,
                     color=p_colors, edgecolor='white', linewidth=1.5)
axes[1].axvline(x=70, color=GREEN, linestyle='--',
                linewidth=1.5, label='Safe threshold', alpha=0.8)
axes[1].axvline(x=40, color=AMBER, linestyle='--',
                linewidth=1.5, label='Warning threshold', alpha=0.8)
axes[1].set_title('Chart 2: Avg Trust Score by Cloud Platform',
                  fontweight='bold', color=MIDNIGHT, pad=12)
axes[1].set_xlabel('Average Trust Score (%)')
axes[1].set_xlim(0, 110)
axes[1].legend(fontsize=8)
for bar, val in zip(bars2, platform_scores.values):
    axes[1].text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}%', va='center',
                 fontsize=9, fontweight='bold', color=MIDNIGHT)

# ── CHART 3: Access Decision Distribution ─────────────────────────────────────
decision_counts = df['risk'].value_counts()
pie_colors = []
for d in decision_counts.index:
    if 'HIGH'   in d: pie_colors.append(RED)
    elif 'MED'  in d: pie_colors.append(AMBER)
    else:             pie_colors.append(GREEN)

axes[2].pie(decision_counts.values,
            labels=decision_counts.index,
            colors=pie_colors, autopct='%1.0f%%',
            startangle=90,
            wedgeprops={'edgecolor':'white', 'linewidth':2.5},
            textprops={'fontsize':10})
axes[2].set_title('Chart 3: Risk Level Distribution',
                  fontweight='bold', color=MIDNIGHT, pad=12)

plt.tight_layout()
plt.savefig('ztsa_charts_1_2_3.png', dpi=150,
            bbox_inches='tight', facecolor=OFFWHITE)
plt.show()
print('✅ Charts 1, 2 & 3 complete — saved as ztsa_charts_1_2_3.png')

---
## 📊 Cell 6 — Visualisation: Charts 4, 5 & 6

In [ ]:
# ── CELL 6: Charts 4, 5 & 6 ──────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle('ZTSA Cloud Security — Compliance & Threat Analysis',
             fontsize=14, fontweight='bold', color=MIDNIGHT, y=1.02)

# ── CHART 4: Security Factor Compliance ───────────────────────────────────────
labels   = list(compliance.keys())
values   = list(compliance.values())
c_colors = [GREEN if v >= 70 else AMBER if v >= 50 else RED for v in values]

bars = axes[0].barh(labels, values,
                    color=c_colors, edgecolor='white', linewidth=1.5)
axes[0].axvline(x=70, color=GREEN, linestyle='--',
                linewidth=1.5, label='Target (70%)', alpha=0.8)
axes[0].set_title('Chart 4: Security Factor Compliance (%)',
                  fontweight='bold', color=MIDNIGHT, pad=12)
axes[0].set_xlabel('Compliance Rate (%)')
axes[0].set_xlim(0, 115)
axes[0].legend(fontsize=8)
for bar, val in zip(bars, values):
    axes[0].text(bar.get_width() + 1,
                 bar.get_y() + bar.get_height()/2,
                 f'{val:.0f}%', va='center',
                 fontsize=8, fontweight='bold', color=MIDNIGHT)

# ── CHART 5: Cloud Breach Trends (IBM/ENISA Real Data) ────────────────────────
years  = ['2020', '2021', '2022', '2023', '2024']
cost   = [3.86,   4.24,   4.35,   4.75,   4.90]  # IBM breach cost $M
cloud  = [43,     50,     60,     72,      82]    # % involving cloud IBM 2024

ax5a = axes[1]
ax5b = ax5a.twinx()
line1 = ax5a.plot(years, cost, color=DEEP, linewidth=2.5,
                  marker='o', markersize=7, label='Breach Cost ($M)')
line2 = ax5b.plot(years, cloud, color=RED, linewidth=2.5,
                  marker='s', markersize=7, linestyle='--',
                  label='Cloud Breaches (%)')
ax5a.fill_between(years, cost, alpha=0.08, color=DEEP)
ax5a.set_title('Chart 5: Cloud Breach Trends 2020–2024\n(IBM Cost of Breach Report)',
               fontweight='bold', color=MIDNIGHT, pad=12)
ax5a.set_ylabel('Average Breach Cost ($M)', color=DEEP)
ax5b.set_ylabel('% Involving Cloud Assets', color=RED)
lines = line1 + line2
labs  = [l.get_label() for l in lines]
ax5a.legend(lines, labs, fontsize=8, loc='upper left')
for i, (yr, c) in enumerate(zip(years, cost)):
    ax5a.text(i, c + 0.05, f'${c}M', ha='center',
              fontsize=7.5, color=DEEP, fontweight='bold')

# ── CHART 6: Live ZTSA Dashboard ──────────────────────────────────────────────
bar_colors6 = []
for score in df['score']:
    if score >= 70:   bar_colors6.append(GREEN)
    elif score >= 40: bar_colors6.append(AMBER)
    else:             bar_colors6.append(RED)

bars6 = axes[2].bar(df['name'], df['score'],
                    color=bar_colors6, edgecolor='white',
                    linewidth=1.5, width=0.65)
axes[2].axhline(y=70, color=GREEN, linestyle='--',
                linewidth=1.5, alpha=0.8)
axes[2].axhline(y=40, color=AMBER, linestyle='--',
                linewidth=1.5, alpha=0.8)
axes[2].set_title('Chart 6: Live ZTSA Access Dashboard',
                  fontweight='bold', color=MIDNIGHT, pad=12)
axes[2].set_ylabel('ZTSA Trust Score (%)')
axes[2].set_ylim(0, 125)
axes[2].tick_params(axis='x', rotation=45)

for bar, val, r in zip(bars6, df['score'], results):
    if val >= 70:   emoji = '✅'
    elif val >= 40: emoji = '⚠️'
    else:           emoji = '🚨'
    axes[2].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1.5,
                 f'{emoji}\n{val}%',
                 ha='center', fontsize=8,
                 fontweight='bold', color=MIDNIGHT)

granted_p = mpatches.Patch(color=GREEN, label='✅ Access Granted')
verify_p  = mpatches.Patch(color=AMBER, label='⚠️  Verify Identity')
denied_p  = mpatches.Patch(color=RED,   label='🚨 Access Denied')
axes[2].legend(handles=[granted_p, verify_p, denied_p], fontsize=8)

plt.tight_layout()
plt.savefig('ztsa_charts_4_5_6.png', dpi=150,
            bbox_inches='tight', facecolor=OFFWHITE)
plt.show()
print('✅ Charts 4, 5 & 6 complete — saved as ztsa_charts_4_5_6.png')

---
## 🔴 Cell 7 — Live Demo: Test Any Cloud User

**This is the live demo cell.** Change the values below and re-run to see the ZTSA score change instantly.

> **Try this during demo:** Change `mfa_enabled` to `False` and `s3_public_bucket` to `True` — watch the score drop!

In [ ]:
# ── CELL 7: Live ZTSA Demo — Test Any Cloud User ──────────────────────────────
# 👆 CHANGE THESE VALUES TO SEE THE ZTSA SCORE CHANGE LIVE!

test_user = {
    # ── IDENTITY ────────────────────────────────────
    'mfa_enabled':        True,   # Try changing to False
    'sso_configured':     True,
    'valid_certificate':  True,

    # ── DEVICE HEALTH ────────────────────────────────
    'device_patched':     True,
    'device_encrypted':   True,
    'antivirus_active':   True,
    'mdm_enrolled':       True,

    # ── CLOUD CONFIGURATION ──────────────────────────
    's3_public_bucket':   False,  # Try changing to True
    'iam_overprivileged': False,
    'cloudtrail_enabled': True,

    # ── NETWORK ──────────────────────────────────────
    'vpn_connected':      True,
    'known_ip':           True,
    'geo_anomaly':        False,

    # ── BEHAVIOUR ────────────────────────────────────
    'failed_logins':      0,      # Try changing to 6
    'unusual_hours':      False,
    'high_data_volume':   False,
}

# Run ZTSA analysis
result = ztsa_score(test_user)

# ── Print detailed results ─────────────────────────────────────────────────────
print('=' * 60)
print('   🔐  ZTSA LIVE CLOUD SECURITY ASSESSMENT')
print('=' * 60)
print(f'  Trust Score  : {result["score"]}%')
print(f'  Risk Level   : {result["risk"]}')
print(f'  Decision     : {result["decision"]}')
if result['risks']:
    print(f'  Risk Areas   : {", ".join(result["risks"])}')
print('-' * 60)
print('  DETAILED SECURITY FINDINGS:')
for finding in result['findings']:
    print(f'    {finding}')
print('=' * 60)

# ── Visual score gauge ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(1, 1, figsize=(10, 4))
score = result['score']

if score >= 70:   bar_col = GREEN;  status = '✅ ACCESS GRANTED'
elif score >= 40: bar_col = AMBER;  status = '⚠️  VERIFY IDENTITY'
else:             bar_col = RED;    status = '🚨 ACCESS DENIED'

# Background zones
ax.barh([''], [40],   color=RED,   alpha=0.15, height=0.4)
ax.barh([''], [30],   color=AMBER, alpha=0.15, height=0.4, left=40)
ax.barh([''], [30],   color=GREEN, alpha=0.15, height=0.4, left=70)
# Score bar
ax.barh([''], [score], color=bar_col, alpha=0.9,  height=0.35)

ax.axvline(x=70, color=GREEN, linestyle='--', linewidth=2, alpha=0.8)
ax.axvline(x=40, color=AMBER, linestyle='--', linewidth=2, alpha=0.8)
ax.set_xlim(0, 105)
ax.set_title(f'ZTSA Trust Score: {score}%  —  {status}',
             fontsize=13, fontweight='bold', color=MIDNIGHT)
ax.text(score + 1.5, 0, f'{score}%',
        va='center', fontsize=14,
        fontweight='bold', color=MIDNIGHT)
ax.text(40, -0.35, 'DENIED\n<40%', ha='center',
        fontsize=9, color=RED, fontweight='bold')
ax.text(55, -0.35, 'VERIFY\n40-69%', ha='center',
        fontsize=9, color=AMBER, fontweight='bold')
ax.text(85, -0.35, 'GRANTED\n≥70%', ha='center',
        fontsize=9, color=GREEN, fontweight='bold')
ax.set_yticks([])
ax.set_xlabel('ZTSA Trust Score (%)', fontsize=11)

plt.tight_layout()
plt.savefig('ztsa_live_demo.png', dpi=150,
            bbox_inches='tight', facecolor=OFFWHITE)
plt.show()
print('✅ Live demo complete — try changing values above and re-running!')

---
## 📋 Cell 8 — Final Summary Report

In [ ]:
# ── CELL 8: Final Summary Report ──────────────────────────────────────────────

granted = len(df[df['score'] >= 70])
verify  = len(df[(df['score'] >= 40) & (df['score'] < 70)])
denied  = len(df[df['score'] < 40])

# Lowest compliance factors
lowest = sorted(compliance.items(), key=lambda x: x[1])[:3]

print('=' * 65)
print('   🔐  ZTSA CLOUD SECURITY — FINAL EVALUATION REPORT')
print('   MSc Cyber Security | Ulster University Belfast')
print('=' * 65)
print(f'  Framework          : NIST SP 800-207 Zero Trust Architecture')
print(f'  Cloud Platforms    : AWS, Azure, GCP, Multi-Cloud')
print(f'  Users Assessed     : {len(df)}')
print(f'  Security Factors   : 15 (Identity, Device, Cloud, Network, Behaviour)')
print('-' * 65)
print(f'  ✅ Access Granted  : {granted} users ({granted/len(df)*100:.0f}%)')
print(f'  ⚠️  Verify Identity : {verify} users ({verify/len(df)*100:.0f}%)')
print(f'  🚨 Access Denied   : {denied} users ({denied/len(df)*100:.0f}%)')
print('-' * 65)
print(f'  Average Trust Score: {df["score"].mean():.1f}%')
print(f'  Score Range        : {df["score"].min()}% — {df["score"].max()}% ({df["score"].max()-df["score"].min()} point gap)')
print(f'  Std Deviation      : {df["score"].std():.1f} (high = good discrimination)')
print('-' * 65)
print('  ⚠️  TOP 3 LOWEST COMPLIANCE AREAS (Priority Remediation):')
for i, (factor, rate) in enumerate(lowest, 1):
    print(f'    {i}. {factor:<25} {rate:.0f}% compliance')
print('-' * 65)
print('  🔴 CRITICAL CLOUD MISCONFIGURATIONS FOUND:')
print(f'    Public S3 Buckets     : {risk_flags["s3_public_bucket"]} users affected')
print(f'    IAM Over-Privileged   : {risk_flags["iam_overprivileged"]} users affected')
print(f'    Geo-Location Anomaly  : {risk_flags["geo_anomaly"]} users affected')
print(f'    Unusual Access Hours  : {risk_flags["unusual_hours"]} users affected')
print(f'    High Data Volume      : {risk_flags["high_data_volume"]} users affected')
print('-' * 65)
print('  Charts saved:')
print('    ztsa_charts_1_2_3.png  — Trust scores, platform risk, distribution')
print('    ztsa_charts_4_5_6.png  — Compliance, breach trends, live dashboard')
print('    ztsa_live_demo.png     — Live score gauge')
print('=' * 65)
print('  ✅ ZTSA Cloud Security Analyser completed successfully!')
print('=' * 65)